# Understanding Frontmatter in Python

This notebook demonstrates how to parse **Frontmatter**, a popular documentation format commonly used by modern static site generators and frameworks like Jekyll, Hugo, and Next.js. 

### What is Frontmatter?
Frontmatter is a way to attach structured metadata to a Markdown document. The section between the `---` markers contains **YAML metadata** that describes the document, while everything below the markers is the regular **Markdown content**. This allows us to extract structured information (like titles, tags, and difficulty levels) programmatically alongside the main text.

In [1]:
# First, let's install the python-frontmatter library
!pip install python-frontmatter

In [2]:
# Create a sample markdown file with frontmatter to practice with
sample_content = """---
title: "Getting Started with AI"
author: "John Doe"
date: "2024-01-15"
tags: ["ai", "machine-learning", "tutorial"]
difficulty: "beginner"
---
# Getting Started with AI

This is the main content of the document written in **Markdown**.

You can include code blocks, links, and other formatting here."""

# Save this to a local file
with open('example.md', 'w', encoding='utf-8') as f:
    f.write(sample_content)

print("Sample file 'example.md' created successfully!")

Sample file 'example.md' created successfully!


### Reading and Parsing the Frontmatter

Now, we will use the `frontmatter` library to load the file and separate the metadata from the actual content.

In [3]:
import frontmatter

# Load the markdown file
with open('example.md', 'r', encoding='utf-8') as f:
    post = frontmatter.load(f)

# Access specific metadata keys
print("--- Metadata Extraction ---")
print(f"Title: {post.metadata['title']}")
print(f"Tags:  {post.metadata['tags']}")

print("\n--- Content Extraction ---")
# Access the markdown content (excluding the frontmatter block)
print(post.content)

--- Metadata Extraction ---
Title: Getting Started with AI
Tags:  ['ai', 'machine-learning', 'tutorial']

--- Content Extraction ---
# Getting Started with AI

This is the main content of the document written in **Markdown**.

You can include code blocks, links, and other formatting here.


### Exporting to a Dictionary

If you need to work with all of the data at once (for example, saving it to a database or converting it to JSON), you can easily dump both the metadata and content together using the `.to_dict()` method.

In [6]:
# Convert the entire parsed post into a single dictionary
post_dict = post.to_dict()

# Display the dictionary
import pprint
pprint.pprint(post_dict)

{'author': 'John Doe',
 'content': '# Getting Started with AI\n'
            '\n'
            'This is the main content of the document written in '
            '**Markdown**.\n'
            '\n'
            'You can include code blocks, links, and other formatting here.',
 'date': '2024-01-15',
 'difficulty': 'beginner',
 'tags': ['ai', 'machine-learning', 'tutorial'],
 'title': 'Getting Started with AI'}


## Complete Implementation: Processing Multiple Files from GitHub

## Complete Implementation: Processing Multiple Files from GitHub

Now that we know how to process a single Markdown file, let's scale this up. In a real-world scenario (like building a search engine or an AI knowledge base), you will need to process multiple files from a repository.

We will work with the FAQ repository from **DataTalks.Club**:
* GitHub Repository: `https://github.com/DataTalksClub/faq`

### The Strategy: In-Memory Zip Processing
Instead of cloning the repository using Git and saving files to our hard drive, we can download the entire repository as a `.zip` archive directly into our computer's memory. 

**The Plan:**
1. Use `requests` to download the zip archive from GitHub.
2. Open the archive in-memory using Python's built-in `zipfile` and `io` modules.
3. Iterate over all `.md` and `.mdx` files in the repo.
4. Extract the frontmatter and content, saving everything into a structured list.

In [11]:
# Import the required libraries
import io
import zipfile
import requests
import frontmatter
import pprint

### Step 1: Download the Repository Zip File
GitHub provides a convenient URL format (`/zip/refs/heads/main`) to download the latest state of a repository's main branch.

In [13]:
# GitHub zip download URL
url = 'https://codeload.github.com/DataTalksClub/faq/zip/refs/heads/main'

print("Downloading repository zip archive...")
resp = requests.get(url)

# Check if the download was successful
if resp.status_code == 200:
    print(f"Successfully downloaded! File size: {len(resp.content)} bytes")
else:
    print(f"Failed to download. Status code: {resp.status_code}")

Successfully downloaded! File size: 26211633 bytes


### Step 2: Extract and Parse Markdown Files In-Memory
We use `io.BytesIO` to treat the downloaded bytes as a file-like object, allowing `zipfile.ZipFile` to read it without saving it to disk. 

*Note: We check for both `.md` and `.mdx` extensions to ensure we capture standard Markdown as well as React Markdown formats.*

In [15]:
repository_data = []

# Create a ZipFile object from the downloaded content
with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
    for file_info in zf.infolist():
        filename = file_info.filename.lower()

        # Only process standard markdown (.md) and React markdown (.mdx) files
        if not (filename.endswith('.md') or filename.endswith('.mdx')):
            continue

        # Read and parse each file
        with zf.open(file_info) as f_in:
            content = f_in.read()
            
            # Use frontmatter.loads() for parsing string/bytes content
            post = frontmatter.loads(content)
            
            # Convert to dictionary and attach the filename for reference
            data = post.to_dict()
            data['filename'] = filename
            
            repository_data.append(data)

print(f"Processing complete! Extracted {len(repository_data)} files.")

Processing complete! Extracted 1216 files.


In [17]:
zf.close()

In [18]:
print(repository_data[1])

{'content': '# FAQ Bot Feedback - PR Review Corrections\n\n## 1. Wrong Section Placement\n\nKestra-related FAQs were incorrectly placed in `general` or `module-1` instead of `module-2` (workflow orchestration):\n\n| PR | Issue | Correction |\n|----|-------|------------|\n| #141 | Kestra IANA timezones | general → module-2, sort_order 20 |\n| #137 | Kestra stdout variables | general → module-2, sort_order 21 |\n| #135 | Kestra outputFiles visibility | general → module-2, sort_order 22 |\n| #118 | Kestra Docker socket | module-1 → module-2, sort_order 23 |\n\n**Rule**: Kestra questions belong in `module-2` (workflow orchestration), not `general` or `module-1`.\n\n---\n\n## 2. Not Relevant for Course (closed)\n\n| PR | Topic | Reason |\n|----|-------|--------|\n| #123 | Installing vim on Ubuntu | Basic Linux admin, outside course scope |\n| #116 | SQL LEFT JOIN returns NULL | Basic SQL concept, not course-specific |\n\n**Rule**: Fundamental tool/SQL concepts that aren\'t course-specific s

### Step 3: Inspect the Ingested Data
Let's look at one of the processed items in our list to verify that the frontmatter metadata, content, and filename were successfully captured.

In [16]:
# Print an item from the repository array to inspect its structure
if len(repository_data) > 1:
    print("Sample Extracted Data:")
    pprint.pprint(repository_data[1])
else:
    print("Not enough files extracted to show index 1.")

Sample Extracted Data:
{'content': '# FAQ Bot Feedback - PR Review Corrections\n'
            '\n'
            '## 1. Wrong Section Placement\n'
            '\n'
            'Kestra-related FAQs were incorrectly placed in `general` or '
            '`module-1` instead of `module-2` (workflow orchestration):\n'
            '\n'
            '| PR | Issue | Correction |\n'
            '|----|-------|------------|\n'
            '| #141 | Kestra IANA timezones | general → module-2, sort_order '
            '20 |\n'
            '| #137 | Kestra stdout variables | general → module-2, sort_order '
            '21 |\n'
            '| #135 | Kestra outputFiles visibility | general → module-2, '
            'sort_order 22 |\n'
            '| #118 | Kestra Docker socket | module-1 → module-2, sort_order '
            '23 |\n'
            '\n'
            '**Rule**: Kestra questions belong in `module-2` (workflow '
            'orchestration), not `general` or `module-1`.\n'
            '\n'
     

## Packaging Everything Into a Reusable Function

Now that we have successfully tested the logic step-by-step, let's refactor our code into a clean, reusable function. This will allow us to easily ingest Markdown documentation from *any* public GitHub repository.

We will add a few improvements here:
1. **Error Handling:** A `try-except` block to skip a single corrupted file without crashing the entire download process.
2. **Text Decoding:** Explicitly decoding the file bytes using `utf-8` with `errors='ignore'` to avoid encoding crashes.

In [20]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com'
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)

    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    
    # Open the zip file from memory
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        for file_info in zf.infolist():
            filename = file_info.filename
            filename_lower = filename.lower()

            # Filter for markdown files
            if not (filename_lower.endswith('.md') or filename_lower.endswith('.mdx')):
                continue

            try:
                with zf.open(file_info) as f_in:
                    # Decode bytes to string safely
                    content = f_in.read().decode('utf-8', errors='ignore')
                    
                    # Parse frontmatter string
                    post = frontmatter.loads(content)
                    
                    # Package into a dictionary
                    data = post.to_dict()
                    data['filename'] = filename
                    repository_data.append(data)
                    
            except Exception as e:
                print(f"Error processing {filename}: {e}")
                continue

    return repository_data

### Testing the Function with Multiple Repositories

Let's use our new function to fetch data from two completely different documentation sources: the **DataTalks.Club FAQ** and the **Evidently AI Documentation**.

In [21]:
# Fetch DataTalks.Club FAQ
print("Fetching DataTalksClub FAQ...")
dtc_faq = read_repo_data('DataTalksClub', 'faq')

# Fetch Evidently AI Docs
print("Fetching Evidently AI Docs...")
evidently_docs = read_repo_data('evidentlyai', 'docs')

print("\n--- Ingestion Results ---")
print(f"DataTalksClub FAQ documents: {len(dtc_faq)}")
print(f"Evidently AI documents:     {len(evidently_docs)}")

Fetching DataTalksClub FAQ...
Fetching Evidently AI Docs...

--- Ingestion Results ---
DataTalksClub FAQ documents: 1216
Evidently AI documents:     95
